In [1]:
%pip install -q "lightgbm>=4,<5" "wandb>=0.19,<1" "optuna>=4,<5"


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import wandb

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
VALIDATION_WEEKS = 32
HOLIDAY_WEIGHT = 5
SEED = 42

wandb.login()

df_train = pd.read_csv("/content/drive/My Drive/walmart_competition_data/train.csv", parse_dates=["Date"])
df_test = pd.read_csv("/content/drive/My Drive/walmart_competition_data/test.csv", parse_dates=["Date"])
df_features = pd.read_csv("/content/drive/My Drive/walmart_competition_data/features.csv", parse_dates=["Date"])
df_stores = pd.read_csv("/content/drive/My Drive/walmart_competition_data/stores.csv")

df_train_merged = df_train.merge(df_stores, on="Store", how="left")
df_train_merged = df_train_merged.merge(
    df_features,
    on=["Store", "Date", "IsHoliday"],
    how="left",
)

# Time-based validation split: sort chronologically and use the last 32 weekly dates as validation.
df_train_merged = df_train_merged.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
validation_dates = np.sort(df_train_merged["Date"].unique())[-VALIDATION_WEEKS:]

train_df_split = df_train_merged.loc[~df_train_merged["Date"].isin(validation_dates)].copy()
val_df_split = df_train_merged.loc[df_train_merged["Date"].isin(validation_dates)].copy()

train_df_split = train_df_split.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
val_df_split = val_df_split.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

y_train = train_df_split["Weekly_Sales"]
is_holiday_train = train_df_split["IsHoliday"]
X_train = train_df_split.copy()

y_val = val_df_split["Weekly_Sales"]
is_holiday_val = val_df_split["IsHoliday"]
X_val = val_df_split.copy()

split_summary = {
    "validation_weeks": VALIDATION_WEEKS,
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "train_start": str(X_train["Date"].min().date()),
    "train_end": str(X_train["Date"].max().date()),
    "validation_start": str(X_val["Date"].min().date()),
    "validation_end": str(X_val["Date"].max().date()),
    "validation_unique_weeks": int(X_val["Date"].nunique()),
}

print(f"Train dates: {X_train['Date'].min().date()} to {X_train['Date'].max().date()}")
print(f"Validation dates: {X_val['Date'].min().date()} to {X_val['Date'].max().date()}")
print(f"Validation unique weeks: {X_val['Date'].nunique()}")
print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nmetr23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train dates: 2010-02-05 to 2012-03-16
Validation dates: 2012-03-23 to 2012-10-26
Validation unique weeks: 32
Train shape: (326856, 16)
Validation shape: (94714, 16)


In [5]:
X_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,2,2010-02-05,50605.27,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
2,1,3,2010-02-05,13740.12,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
3,1,4,2010-02-05,39954.04,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
4,1,5,2010-02-05,32229.38,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106


In [6]:
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


MARKDOWN_COLS = ("MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5")
NUMERIC_EXTERNAL_COLS = ("CPI", "Unemployment", "Temperature", "Fuel_Price")


def _existing_columns(frame: pd.DataFrame, columns: Iterable[str]) -> list[str]:
    return [col for col in columns if col in frame.columns]


class WalmartFeatureCleaner(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        numeric_impute_cols: tuple[str, ...] = NUMERIC_EXTERNAL_COLS,
        add_markdown_missing_indicators: bool = True,
        markdown_fill_value: float = 0.0,
        numeric_impute_strategy: str = "median",
        category_cols: tuple[str, ...] = ("Store", "Dept", "Type"),
    ):
        self.date_col = date_col
        self.markdown_cols = markdown_cols
        self.numeric_impute_cols = numeric_impute_cols
        self.add_markdown_missing_indicators = add_markdown_missing_indicators
        self.markdown_fill_value = markdown_fill_value
        self.numeric_impute_strategy = numeric_impute_strategy
        self.category_cols = category_cols

    def fit(self, X: pd.DataFrame, y=None):
        if self.numeric_impute_strategy not in {"median", "mean", "none"}:
            raise ValueError("numeric_impute_strategy must be 'median', 'mean', or 'none'.")

        self.numeric_fill_values_ = {}
        numeric_cols = _existing_columns(X, self.numeric_impute_cols)
        if self.numeric_impute_strategy != "none":
            for col in numeric_cols:
                if self.numeric_impute_strategy == "median":
                    self.numeric_fill_values_[col] = X[col].median()
                else:
                    self.numeric_fill_values_[col] = X[col].mean()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        if self.date_col in frame.columns:
            frame[self.date_col] = pd.to_datetime(frame[self.date_col])

        for col in _existing_columns(frame, self.markdown_cols):
            if self.add_markdown_missing_indicators:
                frame[f"{col}_missing"] = frame[col].isna().astype("int8")
            frame[col] = frame[col].fillna(self.markdown_fill_value)

        for col, value in getattr(self, "numeric_fill_values_", {}).items():
            if col in frame.columns:
                frame[col] = frame[col].fillna(value)

        for col in _existing_columns(frame, self.category_cols):
            frame[col] = frame[col].astype("category")

        if "IsHoliday" in frame.columns:
            frame["IsHoliday"] = frame["IsHoliday"].astype("int8")

        return frame


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        start_date: str = "2010-02-05",
        add_cyclical_features: bool = True,
        drop_date: bool = False,
    ):
        self.date_col = date_col
        self.start_date = start_date
        self.add_cyclical_features = add_cyclical_features
        self.drop_date = drop_date

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])
        iso = date.dt.isocalendar()

        frame["Year"] = date.dt.year.astype("int16")
        frame["Month"] = date.dt.month.astype("int8")
        frame["WeekOfYear"] = iso.week.astype("int8")
        frame["Quarter"] = date.dt.quarter.astype("int8")
        frame["DayOfYear"] = date.dt.dayofyear.astype("int16")
        frame["DaysFromStart"] = (date - pd.Timestamp(self.start_date)).dt.days.astype("int16")

        if self.add_cyclical_features:
            frame["WeekSin"] = np.sin(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["WeekCos"] = np.cos(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["MonthSin"] = np.sin(2 * np.pi * frame["Month"] / 12.0)
            frame["MonthCos"] = np.cos(2 * np.pi * frame["Month"] / 12.0)

        if self.drop_date:
            frame = frame.drop(columns=[self.date_col])

        return frame


class WalmartHolidayFeatureTransformer(BaseEstimator, TransformerMixin):

    HOLIDAY_DATES = {
        "SuperBowl": ("2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"),
        "LaborDay": ("2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"),
        "Thanksgiving": ("2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"),
        "Christmas": ("2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"),
    }

    def __init__(
        self,
        date_col: str = "Date",
        add_holiday_flags: bool = True,
        add_proximity_features: bool = True,
    ):
        self.date_col = date_col
        self.add_holiday_flags = add_holiday_flags
        self.add_proximity_features = add_proximity_features

    def fit(self, X: pd.DataFrame, y=None):
        self.holiday_dates_ = {
            name: pd.to_datetime(list(dates)) for name, dates in self.HOLIDAY_DATES.items()
        }
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])

        for name, holiday_dates in self.holiday_dates_.items():
            if self.add_holiday_flags:
                frame[f"Is{name}Week"] = date.isin(holiday_dates).astype("int8")

            if self.add_proximity_features:
                distances = np.vstack([(date - holiday).dt.days.to_numpy() for holiday in holiday_dates])
                nearest_distance = distances[np.abs(distances).argmin(axis=0), np.arange(len(date))]
                frame[f"DaysToNearest{name}"] = np.abs(nearest_distance).astype("int16")
                frame[f"WeeksToNearest{name}"] = (np.abs(nearest_distance) / 7.0).astype("float32")

        return frame


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        add_total_markdown: bool = True,
        add_has_markdown: bool = True,
        add_log_markdowns: bool = True,
        add_holiday_interaction: bool = True,
        holiday_col: str = "IsHoliday",
    ):
        self.markdown_cols = markdown_cols
        self.add_total_markdown = add_total_markdown
        self.add_has_markdown = add_has_markdown
        self.add_log_markdowns = add_log_markdowns
        self.add_holiday_interaction = add_holiday_interaction
        self.holiday_col = holiday_col

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        markdown_cols = _existing_columns(frame, self.markdown_cols)

        if self.add_total_markdown and markdown_cols:
            frame["TotalMarkDown"] = frame[markdown_cols].sum(axis=1)

        if self.add_has_markdown:
            for col in markdown_cols:
                frame[f"Has{col}"] = (frame[col] > 0).astype("int8")
            if "TotalMarkDown" in frame.columns:
                frame["HasAnyMarkDown"] = (frame["TotalMarkDown"] > 0).astype("int8")

        if self.add_log_markdowns:
            for col in markdown_cols:
                frame[f"{col}_log1p"] = np.log1p(frame[col].clip(lower=0))
            if "TotalMarkDown" in frame.columns:
                frame["TotalMarkDown_log1p"] = np.log1p(frame["TotalMarkDown"].clip(lower=0))

        if self.add_holiday_interaction and self.holiday_col in frame.columns:
            if "TotalMarkDown" in frame.columns:
                frame["Holiday_TotalMarkDown"] = frame[self.holiday_col] * frame["TotalMarkDown"]
            for col in markdown_cols:
                frame[f"Holiday_{col}"] = frame[self.holiday_col] * frame[col]

        return frame


class InteractionFeatureTransformer(BaseEstimator, TransformerMixin):


    def __init__(
        self,
        interactions: tuple[tuple[str, ...], ...] = (("Store", "Dept"), ("Type", "Dept")),
        separator: str = "_",
        as_category: bool = True,
    ):
        self.interactions = interactions
        self.separator = separator
        self.as_category = as_category

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        for cols in self.interactions:
            if all(col in frame.columns for col in cols):
                new_col = self.separator.join(cols)
                values = frame[list(cols)].astype(str).agg(self.separator.join, axis=1)
                frame[new_col] = values.astype("category") if self.as_category else values

        return frame


class LagRollingFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        group_cols: tuple[str, ...] = ("Store", "Dept"),
        date_col: str = "Date",
        target_col: str = "Weekly_Sales",
        lags: tuple[int, ...] = (1, 4, 13, 52),
        rolling_windows: tuple[int, ...] = (4, 13),
        rolling_stats: tuple[str, ...] = ("mean", "std"),
        min_periods: int = 1,
    ):
        self.group_cols = group_cols
        self.date_col = date_col
        self.target_col = target_col
        self.lags = lags
        self.rolling_windows = rolling_windows
        self.rolling_stats = rolling_stats
        self.min_periods = min_periods

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.target_col not in X.columns:
            raise ValueError(
                f"{self.target_col!r} is required for lag/rolling features. "
                "For test data, append historical sales first or use recursive inference."
            )

        frame = X.copy().sort_values(list(self.group_cols) + [self.date_col])
        grouped = frame.groupby(list(self.group_cols), observed=True)[self.target_col]

        for lag in self.lags:
            frame[f"lag_{lag}"] = grouped.shift(lag)

        for window in self.rolling_windows:
            shifted = grouped.shift(1)
            rolling = shifted.groupby([frame[col] for col in self.group_cols], observed=True).rolling(
                window=window,
                min_periods=self.min_periods,
            )
            if "mean" in self.rolling_stats:
                frame[f"rolling_mean_{window}"] = rolling.mean().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "std" in self.rolling_stats:
                frame[f"rolling_std_{window}"] = rolling.std().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "min" in self.rolling_stats:
                frame[f"rolling_min_{window}"] = rolling.min().reset_index(level=list(range(len(self.group_cols))), drop=True)
            if "max" in self.rolling_stats:
                frame[f"rolling_max_{window}"] = rolling.max().reset_index(level=list(range(len(self.group_cols))), drop=True)

        return frame.sort_index()


class HistoricalAggregateTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        groupings: tuple[tuple[str, ...], ...] = (("Store",), ("Dept",), ("Store", "Dept"), ("Type", "Dept")),
        target_col: str = "Weekly_Sales",
        stats: tuple[str, ...] = ("mean", "median", "std"),
        fill_missing_with_global: bool = True,
    ):
        self.groupings = groupings
        self.target_col = target_col
        self.stats = stats
        self.fill_missing_with_global = fill_missing_with_global

    def fit(self, X: pd.DataFrame, y=None):
        if self.target_col not in X.columns:
            raise ValueError(f"{self.target_col!r} must be present when fitting aggregates.")

        self.global_stats_ = X[self.target_col].agg(list(self.stats)).to_dict()
        self.aggregate_frames_ = []

        for grouping in self.groupings:
            existing_grouping = tuple(col for col in grouping if col in X.columns)
            if not existing_grouping:
                continue
            prefix = "_".join(existing_grouping)
            agg = (
                X.groupby(list(existing_grouping), observed=True)[self.target_col]
                .agg(list(self.stats))
                .reset_index()
            )
            rename = {stat: f"{prefix}_{self.target_col}_{stat}" for stat in self.stats}
            agg = agg.rename(columns=rename)
            self.aggregate_frames_.append((existing_grouping, agg, rename))

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        for grouping, agg, rename in self.aggregate_frames_:
            frame = frame.merge(agg, on=list(grouping), how="left", validate="many_to_one")
            if self.fill_missing_with_global:
                for stat, col in rename.items():
                    frame[col] = frame[col].fillna(self.global_stats_[stat])

        return frame


class ColumnDropper(BaseEstimator, TransformerMixin):

    def __init__(self, columns: tuple[str, ...] = ("Date", "Weekly_Sales"), errors: str = "ignore"):
        self.columns = columns
        self.errors = errors

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=list(self.columns), errors=self.errors)


class FeatureImportanceSelector(BaseEstimator, TransformerMixin):

    def __init__(self, estimator, threshold: float = 0.0, fit_params: dict | None = None):
        self.estimator = estimator
        self.threshold = threshold
        self.fit_params = fit_params

    def fit(self, X: pd.DataFrame, y):
        fit_params = self.fit_params or {}
        self.estimator.fit(X, y, **fit_params)
        importances = getattr(self.estimator, "feature_importances_", None)
        if importances is None:
            raise ValueError("estimator must expose feature_importances_ after fit.")

        self.feature_importances_ = pd.Series(importances, index=X.columns).sort_values(ascending=False)
        self.selected_features_ = self.feature_importances_[
            self.feature_importances_ > self.threshold
        ].index.tolist()
        if not self.selected_features_:
            raise ValueError("No features passed the importance threshold.")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[self.selected_features_].copy()


def make_walmart_lgbm_feature_pipeline(
    include_lag_features: bool = True,
    drop_target_and_date: bool = True,
) -> Pipeline:

    pre_processing = [
        ("clean", WalmartFeatureCleaner()),
        ("calendar", CalendarFeatureTransformer()),
        ("holiday", WalmartHolidayFeatureTransformer()),
        ("markdown", MarkdownFeatureTransformer()),
        ("interactions", InteractionFeatureTransformer()),
        ("aggregates", HistoricalAggregateTransformer()),
    ]

    if include_lag_features:
        pre_processing.append(("lags_rollings", LagRollingFeatureTransformer()))

    if drop_target_and_date:
        pre_processing.append(("drop_columns", ColumnDropper()))

    return Pipeline(pre_processing)


In [8]:
import optuna
import wandb
import lightgbm as lgb
import matplotlib.pyplot as plt
from wandb.integration.lightgbm import log_summary, wandb_callback

feature_pipeline = make_walmart_lgbm_feature_pipeline(
    include_lag_features=True,
    drop_target_and_date=True
)

X_train_transformed = feature_pipeline.fit_transform(X_train, y_train)
X_val_transformed = feature_pipeline.transform(X_val)

sample_weights_train = np.where(is_holiday_train, HOLIDAY_WEIGHT, 1)
sample_weights_val = np.where(is_holiday_val, HOLIDAY_WEIGHT, 1)

categorical_features = X_train_transformed.select_dtypes(include="category").columns.tolist()

base_params = {
    "objective": "mae",
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "n_estimators": 100, # Fixed number of estimators
}

def objective(trial):
    # Suggest hyperparameters (n_estimators is now fixed in base_params)
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 256),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 0.1),
    }

    model_params = {**base_params, **params}

    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        job_type="train",
        name=f"lightgbm-optuna-trial-{trial.number}",
        tags=["lightgbm", "optuna", "tpe", "time-split"],
        config={
            **split_summary,
            "holiday_weight": HOLIDAY_WEIGHT,
            "feature_count": X_train_transformed.shape[1],
            "categorical_features": categorical_features,
            **model_params,
        },
        reinit=True,

    )

    print(f"\nTraining LightGBM model with Optuna trial {trial.number}: {params}")
    lgbm = lgb.LGBMRegressor(**model_params)

    callbacks = [
        wandb_callback(),
        lgb.log_evaluation(period=25),
    ]

    lgbm.fit(
        X_train_transformed,
        y_train,
        sample_weight=sample_weights_train,
        eval_set=[
            (X_train_transformed, y_train),
            (X_val_transformed, y_val),
        ],
        eval_names=["train", "validation"],
        eval_sample_weight=[sample_weights_train, sample_weights_val],
        eval_metric="mae",
        categorical_feature=categorical_features,
        callbacks=callbacks,
    )
    log_summary(lgbm.booster_, save_model_checkpoint=False)

    y_pred_val = lgbm.predict(X_val_transformed)
    weighted_mae = np.sum(np.abs(y_val - y_pred_val) * sample_weights_val) / np.sum(sample_weights_val)
    mae = np.mean(np.abs(y_val - y_pred_val))
    print(f"Validation Weighted MAE: {weighted_mae:.4f}")

    feature_importance = pd.DataFrame({
        "feature": X_train_transformed.columns,
        "importance": lgbm.feature_importances_,
    }).sort_values("importance", ascending=False)


    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(y_val, y_pred_val, alpha=0.3)
    ax.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
    ax.set_xlabel("Actual Weekly Sales")
    ax.set_ylabel("Predicted Weekly Sales")
    ax.set_title(f"Trial {trial.number}: Actual vs. Predicted Weekly Sales")
    wandb.log(
        {
            "validation/weighted_mae": weighted_mae,
            "validation/mae": mae,
            "model/feature_importance": wandb.Table(dataframe=feature_importance.head(50)),
            "plots/actual_vs_predicted": wandb.Image(fig)
        }
    )
    plt.close(fig)

    run.summary["best_validation_weighted_mae"] = weighted_mae
    run.finish()

    return weighted_mae

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n--- Optuna Hyperparameter Tuning Results ---")
print(f"Number of finished trials: {len(study.trials)}")
print(f"Best trial:")

trial = study.best_trial
print(f"  Value: {trial.value:.4f}")
print(f"  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

best_model_params = {**base_params, **study.best_params}
best_model = lgb.LGBMRegressor(**best_model_params)

best_model.fit(
    X_train_transformed,
    y_train,
    sample_weight=sample_weights_train,
    eval_set=[
        (X_train_transformed, y_train),
        (X_val_transformed, y_val),
    ],
    eval_names=["train", "validation"],
    eval_sample_weight=[sample_weights_train, sample_weights_val],
    eval_metric="mae",
    categorical_feature=categorical_features,
    callbacks=[lgb.log_evaluation(period=25)],
)

print("Optuna hyperparameter tuning complete.")

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
[I 2026-07-06 18:43:11,085] A new study created in memory with name: no-name-8424e2f8-c3b7-4fa6-b51c-c3e6fd0390e5


  0%|          | 0/50 [00:00<?, ?it/s]

iteration,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
train_l1,███▇▇▆▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
validation_l1,██▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
iteration,99



Training LightGBM model with Optuna trial 0: {'learning_rate': 0.023688639503640783, 'num_leaves': 245, 'max_depth': 16, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'subsample_freq': 1, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.005808361216819946, 'reg_lambda': 0.08661761457749352}
[25]	train's l1: 8857.45	validation's l1: 8563.17
[50]	train's l1: 5863.32	validation's l1: 5579.7
[75]	train's l1: 4107.46	validation's l1: 3843.98
[100]	train's l1: 3102.05	validation's l1: 2861.46
Validation Weighted MAE: 2861.4556


iteration,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
train_l1,████▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
validation/mae,▁
validation/weighted_mae,▁
validation_l1,██▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
best_iteration,0
best_validation_weighted_mae,2861.45558
iteration,99
validation/mae,2822.08747
validation/weighted_mae,2861.45558


...

iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▇▇▇▇▇▇▇███
train_l1,██▆▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation/mae,▁
validation/weighted_mae,▁
validation_l1,█▇▆▆▅▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_iteration,0
best_validation_weighted_mae,1582.07391
iteration,99
validation/mae,1550.70917
validation/weighted_mae,1582.07391


[I 2026-07-06 19:10:56,554] Trial 48 finished with value: 1582.0739101995562 and parameters: {'learning_rate': 0.08068701466088149, 'num_leaves': 224, 'max_depth': 18, 'min_child_samples': 96, 'subsample': 0.9849187719627207, 'colsample_bytree': 0.8418112605083504, 'reg_alpha': 0.02609419539517788, 'reg_lambda': 0.004701892083891322}. Best is trial 46 with value: 1573.4988468357697.



Training LightGBM model with Optuna trial 49: {'learning_rate': 0.018426932029835672, 'num_leaves': 229, 'max_depth': 18, 'min_child_samples': 95, 'subsample': 0.9169718644705116, 'subsample_freq': 1, 'colsample_bytree': 0.8364504473852455, 'reg_alpha': 0.03385667459681206, 'reg_lambda': 0.00017819196636367253}
[25]	train's l1: 9780.48	validation's l1: 9456.69
[50]	train's l1: 7037.04	validation's l1: 6727.72
[75]	train's l1: 5190.92	validation's l1: 4878.05
[100]	train's l1: 3980.95	validation's l1: 3695.29
Validation Weighted MAE: 3695.2923


iteration,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train_l1,█▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
validation/mae,▁
validation/weighted_mae,▁
validation_l1,██▇▇▇▇▇▆▆▆▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
best_iteration,0
best_validation_weighted_mae,3695.2923
iteration,99
validation/mae,3644.84815
validation/weighted_mae,3695.2923


[I 2026-07-06 19:11:39,031] Trial 49 finished with value: 3695.292304968762 and parameters: {'learning_rate': 0.018426932029835672, 'num_leaves': 229, 'max_depth': 18, 'min_child_samples': 95, 'subsample': 0.9169718644705116, 'colsample_bytree': 0.8364504473852455, 'reg_alpha': 0.03385667459681206, 'reg_lambda': 0.00017819196636367253}. Best is trial 46 with value: 1573.4988468357697.

--- Optuna Hyperparameter Tuning Results ---
Number of finished trials: 50
Best trial:
  Value: 1573.4988
  Params: 
    learning_rate: 0.08117866851143801
    num_leaves: 196
    max_depth: 19
    min_child_samples: 98
    subsample: 0.9817092354210323
    colsample_bytree: 0.8206871721053576
    reg_alpha: 0.00031014058676548666
    reg_lambda: 0.01501347092737337
[25]	train's l1: 3592.21	validation's l1: 3302.43
[50]	train's l1: 1876.66	validation's l1: 1811.37
[75]	train's l1: 1534.96	validation's l1: 1620.28
[100]	train's l1: 1463.57	validation's l1: 1595.64
Optuna hyperparameter tuning complete.
